In [3]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

In [4]:
BASE_DIR = r"C:\Users\SAKTHI\Desktop\SMARTVISION\smartvision_dataset"

TRAIN_DIR = f"{BASE_DIR}/classification/train"
VAL_DIR = f"{BASE_DIR}/classification/val"
TEST_DIR = f"{BASE_DIR}/classification/test"

os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("✅ Setup complete.")
print("Device:", device)

✅ Setup complete.
Device: cuda


In [5]:
# Image Transforms
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    # transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

In [6]:
# Load Dataset
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

print("\nClasses: ", class_names)
print("\n Classes length",num_classes )


Classes:  ['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted_plant', 'stop_sign', 'traffic_light', 'train', 'truck']

 Classes length 26


In [7]:
# Load Pretrained Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = models.ResNet50_Weights.IMAGENET1K_V2
model = models.resnet50(weights=weights)

# Freeze every parameter in the model
for param in model.parameters():
    param.requires_grad = False

print("All parameters frozen.")
print(model)

All parameters frozen.
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_s

In [8]:

all_params = list(model.named_parameters())
# unfreezing the last 20 of these parameter tensors - while keeping the early layers
last_params = all_params[-20:] 

for name, param in last_params:
    param.requires_grad = True
    print("Unfrozen:", name)

Unfrozen: layer4.1.conv1.weight
Unfrozen: layer4.1.bn1.weight
Unfrozen: layer4.1.bn1.bias
Unfrozen: layer4.1.conv2.weight
Unfrozen: layer4.1.bn2.weight
Unfrozen: layer4.1.bn2.bias
Unfrozen: layer4.1.conv3.weight
Unfrozen: layer4.1.bn3.weight
Unfrozen: layer4.1.bn3.bias
Unfrozen: layer4.2.conv1.weight
Unfrozen: layer4.2.bn1.weight
Unfrozen: layer4.2.bn1.bias
Unfrozen: layer4.2.conv2.weight
Unfrozen: layer4.2.bn2.weight
Unfrozen: layer4.2.bn2.bias
Unfrozen: layer4.2.conv3.weight
Unfrozen: layer4.2.bn3.weight
Unfrozen: layer4.2.bn3.bias
Unfrozen: fc.weight
Unfrozen: fc.bias


In [9]:
in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(inplace=True),
    nn.Dropout(0.4),
    nn.Linear(512, 26)
)

# Make sure the new head is trainable
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
)

scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

In [11]:
best_val_loss = float("inf")
epochs_no_improve = 0
best_model_weights = None

In [13]:
BEST_WEIGHTS_PATH = 'models/resnet50_best.pth'
best_val_acc = 0.0  # initialize before training loop

for epoch in range(1, 15 + 1):

    # ----------------- Training phase -----------------
    model.train()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        total += inputs.size(0)

    train_loss = running_loss / total
    train_acc = running_correct / total

    # ----------------- Validation phase -----------------
    model.eval()
    val_running_loss = 0.0
    val_running_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * inputs.size(0)
            val_running_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += inputs.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_running_correct / val_total

    # ----------------- Scheduler step -----------------
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch:02d}/{15} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={current_lr:.2e}"
    )

    # ----------------- Early stopping + save-best check -----------------
    if val_acc > best_val_acc + 1e-4:
        best_val_acc = val_acc
        epochs_no_improve = 0
        best_model_weights = copy.deepcopy(model.state_dict())

        # Save best weights to disk immediately, so we always have the
        # latest "best" version even if training is interrupted.
        torch.save(best_model_weights, BEST_WEIGHTS_PATH)
        print(f"  -> New best val_acc={val_acc:.4f}. Weights saved to '{BEST_WEIGHTS_PATH}'.")
    else:
        epochs_no_improve += 1
        EARLY_STOPPING_PATIENCE = 7
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"\nNo improvement for {EARLY_STOPPING_PATIENCE} epochs "
                  f"-> stopping early at epoch {epoch}.")
            break

Epoch 01/15 | train_loss=2.5813 train_acc=0.3632 | val_loss=1.4247 val_acc=0.6769 | lr=1.00e-04
  -> New best val_acc=0.6769. Weights saved to 'models/resnet50_best.pth'.
Epoch 02/15 | train_loss=1.2131 train_acc=0.6648 | val_loss=0.9105 val_acc=0.7513 | lr=1.00e-04
  -> New best val_acc=0.7513. Weights saved to 'models/resnet50_best.pth'.
Epoch 03/15 | train_loss=0.7489 train_acc=0.7813 | val_loss=0.8169 val_acc=0.7744 | lr=1.00e-04
  -> New best val_acc=0.7744. Weights saved to 'models/resnet50_best.pth'.
Epoch 04/15 | train_loss=0.4952 train_acc=0.8582 | val_loss=0.8072 val_acc=0.7641 | lr=1.00e-04
Epoch 05/15 | train_loss=0.3478 train_acc=0.9027 | val_loss=0.8432 val_acc=0.7692 | lr=1.00e-04
Epoch 06/15 | train_loss=0.2386 train_acc=0.9379 | val_loss=0.8520 val_acc=0.7667 | lr=1.00e-04
Epoch 07/15 | train_loss=0.1692 train_acc=0.9610 | val_loss=0.9908 val_acc=0.7308 | lr=1.00e-04
Epoch 08/15 | train_loss=0.1311 train_acc=0.9665 | val_loss=0.9718 val_acc=0.7590 | lr=5.00e-05
Epoch 0

In [15]:
print(f"Best model weights loaded. Best val_loss = {val_acc:.4f}")
print(f"Saved at: {BEST_WEIGHTS_PATH}")


Best model weights loaded. Best val_loss = 0.7513
Saved at: models/resnet50_best.pth
